# 01 — EDA & Data Preparation

Distribusi dataset, visualisasi sampel, pemeriksaan imbalance, dan pembuatan split stratified.

In [ ]:
# ============================================================
# Setup cell - Kaggle Notebooks (Kaggle-only). Jalankan PALING ATAS.
# Cara attach dataset: panel kanan > + Add Data > cari
#   'fruit and vegetable disease healthy vs rotten' > Add.
# ============================================================
import os
import sys
import shutil
import subprocess
from pathlib import Path

# 1. Clone repo dari GitHub (atau pull jika sudah ada di sesi ini)
REPO_URL = "https://github.com/faizhuda/pcd-kelompok-17.git"
PROJECT_DIR = Path("/kaggle/working/pcd-kelompok-17")
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

# 2. Working directory ke root project + tambah ke sys.path
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# 3. Dependency inti SUDAH pre-installed di Kaggle -> tidak ada pip install.

# 4. Dataset gambar (read-only, hasil + Add Data)
# Auto-detect: Kaggle bisa mount di /kaggle/input/<slug> atau
# /kaggle/input/datasets/<user>/<slug> tergantung cara attach.
_DATASET_SLUG = 'fruit-and-vegetable-disease-healthy-vs-rotten'
_candidates = [
    Path('/kaggle/input') / _DATASET_SLUG,
    Path('/kaggle/input/datasets/muhammad0subhan') / _DATASET_SLUG,
]
RAW_DATA_DIR = next((p for p in _candidates if p.exists()), None)
if RAW_DATA_DIR is None:
    # Fallback: cari folder mana saja di /kaggle/input yang berisi gambar dataset
    for _p in Path('/kaggle/input').rglob(_DATASET_SLUG):
        if _p.is_dir():
            RAW_DATA_DIR = _p
            break
assert RAW_DATA_DIR is not None, "Dataset belum di-attach. + Add Data > cari dataset > Add."

# 5. Auto-restore hasil notebook sebelumnya (untuk notebook 03 & 04).
#    Attach output run lama via: + Add Data > Your Work / Dataset bersama.
def restore_previous_outputs():
    # Kaggle mounts notebook outputs di /kaggle/input/notebooks/<user>/<notebook>/
    # sehingga perlu rglob, bukan glob satu level.
    restored = []
    for repo in Path("/kaggle/input").rglob("pcd-kelompok-17"):
        if not repo.is_dir():
            continue
        for sub in ("results", "data/processed"):
            src_dir = repo / sub
            if src_dir.exists():
                shutil.copytree(src_dir, PROJECT_DIR / sub, dirs_exist_ok=True)
                restored.append(str(src_dir))
    return restored

restored = restore_previous_outputs()
print("Project :", PROJECT_DIR)
print("Dataset :", RAW_DATA_DIR)
print("Restore :", restored or "(mulai dari nol)")


In [ ]:
import os
import sys
from pathlib import Path

# Setup cell sudah chdir ke PROJECT_DIR & menambah sys.path (Kaggle-only).
ROOT = Path("/kaggle/working/pcd-kelompok-17")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.preprocessing import check_integrity
from src.utils import (
    build_dataset_index,
    get_project_paths,
    make_splits,
    save_splits,
    set_seed,
)

set_seed(42)
paths = get_project_paths()
paths


In [ ]:
# RAW_DATA_DIR sudah di-set di setup cell (/kaggle/input/...).
# Tidak ada download: dataset Kaggle di-attach langsung (read-only).
print("Sumber dataset:", RAW_DATA_DIR)


In [ ]:
df = build_dataset_index(RAW_DATA_DIR)
print(f"Total citra: {len(df)}")
print(f"Komoditas: {df['commodity'].nunique()}")
print(f"Label: {df['label'].value_counts().to_dict()}")
df.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["label"].value_counts().plot(kind="bar", ax=axes[0], title="Distribusi Kelas")
df.groupby(["commodity", "label"]).size().unstack(fill_value=0).plot(
    kind="bar", stacked=True, ax=axes[1], title="Komoditas x Kelas"
)
plt.tight_layout()
plt.show()


In [ ]:
import cv2

sample = df.groupby(["commodity", "label"]).first().reset_index().head(6)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    img = cv2.imread(row["filepath"])
    if img is not None:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{row['commodity']} -- {row['label']}")
    ax.axis("off")
plt.suptitle("Contoh Citra")
plt.tight_layout()
plt.show()


In [ ]:
corrupt = [fp for fp in df["filepath"] if not check_integrity(fp)]
print(f"Citra corrupt / tidak terbaca: {len(corrupt)}")


In [ ]:
train, val, test = make_splits(df)
print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
split_path = save_splits(train, val, test, paths["splits"])
print(f"Split disimpan ke: {split_path}")
